# 🗓️ 29일차 스터디 노트북 — 연결 리스트 (포인터로 잇는 자료구조)

**오늘 범위**: 08-1 연결 리스트 · 08-2 포인터를 이용한 연결 리스트 — `Node` / `LinkedList` 클래스 · 삽입·삭제 · `search` · 이터레이터 · 보충수업 8-1, 8-2

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[그림]** 화살표 직접 그리기 · **[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]**

---

## 😵 "외울 게 너무 많다"에 대한 답

오늘 배울 함수가 12개야. `add_first`, `add_last`, `remove_first`, `remove_last`, `remove`, `search`, `clear`, `next`, ... 그리고 각 함수가 `current`를 어떻게 바꾸는지 정리한 **표 8-1**까지.

**그런데 이건 외우는 게 아니야.** 오늘 노트북은 이렇게 설계했어:

| | 외우려 하면 | 대신 이렇게 |
|---|---|---|
| 12개 함수 | 코드를 통째로 암기 | **화살표를 그려서 유도** (PART 3) |
| 표 8-1 (current 값) | 11줄을 암기 | **한 가지 원리**로 전부 유도 (17번) 🔥 |
| 빈/1개/2개 판정식 | `head is None`, `head.next is None`... | **그림 3장**이면 자동으로 나옴 (5번) |

> 🎯 **오늘의 목표: 종이에 화살표를 그릴 수 있으면 코드는 저절로 써진다.**

## 앞으로 달라지는 것

지금까지(1~28일차)는 **배열 안에서** 문제를 풀었어. 오늘부터는 **자료구조 자체를 만든다.**

교재 322p가 배열 리스트의 한계를 이렇게 못 박아:
> **"데이터를 삽입·삭제함에 따라 데이터를 옮겨야 하므로 효율적이지 않습니다."**

## 진행 순서
**개념(1~3) → 상태 판별(4~6) → 삽입·삭제 그림(7~11) → 코드 구현(12~16) → current와 이터레이터(17~19) → 버그와 성능(20~22)**

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
from __future__ import annotations
from typing import Any
import time, sys, io, contextlib

# ===== 교재 실습 8-1 =====
class Node:
    """연결 리스트용 노드 클래스"""
    def __init__(self, data: Any = None, next: Node = None):
        self.data = data        # 데이터
        self.next = next        # 뒤쪽 포인터


class LinkedList:
    """연결 리스트 클래스"""
    def __init__(self) -> None:
        self.no = 0             # 노드의 개수
        self.head = None        # 머리 노드
        self.current = None     # 주목 노드

    def __len__(self) -> int:
        return self.no

    def search(self, data: Any) -> int:
        cnt = 0
        ptr = self.head
        while ptr is not None:
            if ptr.data == data:
                self.current = ptr
                return cnt
            cnt += 1
            ptr = ptr.next
        return -1

    def __contains__(self, data: Any) -> bool:
        return self.search(data) >= 0

    def add_first(self, data: Any) -> None:
        ptr = self.head                          # 삽입 전의 머리 노드
        self.head = self.current = Node(data, ptr)
        self.no += 1

    def add_last(self, data: Any) -> None:
        if self.head is None:
            self.add_first(data)
        else:
            ptr = self.head
            while ptr.next is not None:
                ptr = ptr.next
            ptr.next = self.current = Node(data, None)
            self.no += 1

    def remove_first(self) -> None:
        if self.head is not None:
            self.head = self.current = self.head.next
        self.no -= 1                             # ⚠️ 들여쓰기를 잘 봐! (20번)

    def remove_last(self) -> None:
        if self.head is not None:
            if self.head.next is None:
                self.remove_first()
            else:
                ptr = self.head
                pre = self.head
                while ptr.next is not None:
                    pre = ptr
                    ptr = ptr.next
                pre.next = None
                self.current = pre
                self.no -= 1

    def remove(self, p: Node) -> None:
        if self.head is not None:
            if p is self.head:
                self.remove_first()
            else:
                ptr = self.head
                while ptr.next is not p:
                    ptr = ptr.next
                    if ptr is None:
                        return
                ptr.next = p.next
                self.current = ptr
                self.no -= 1

    def remove_current_node(self) -> None:
        self.remove(self.current)

    def clear(self) -> None:
        while self.head is not None:
            self.remove_first()
        self.current = None
        self.no = 0

    def next(self) -> bool:
        if self.current is None or self.current.next is None:
            return False
        self.current = self.current.next
        return True

    def print_current_node(self) -> None:
        if self.current is None:
            print('주목 노드가 존재하지 않습니다')
        else:
            print(self.current.data)

    def print(self) -> None:
        ptr = self.head
        while ptr is not None:
            print(ptr.data)
            ptr = ptr.next

    def __iter__(self) -> LinkedListIterator:
        return LinkedListIterator(self.head)


class LinkedListIterator:
    """LinkedList의 이터레이터용 클래스"""
    def __init__(self, head: Node):
        self.current = head

    def __iter__(self) -> LinkedListIterator:
        return self

    def __next__(self) -> Any:
        if self.current is None:
            raise StopIteration
        else:
            data = self.current.data
            self.current = self.current.next
            return data


# ===== 실험용 도구 =====
def draw(lst, label=""):
    """연결 리스트를 화살표 그림으로 출력"""
    if label: print(label)
    if lst.head is None:
        print("  head → None   (빈 리스트)")
        print(f"  no={lst.no}, current=None\n"); return
    parts, ptr = [], lst.head
    while ptr is not None:
        mark = "*" if ptr is lst.current else " "
        parts.append(f"[{ptr.data}]{mark}")
        ptr = ptr.next
    cur = lst.current.data if lst.current else None
    print("  head → " + " → ".join(parts) + " → None")
    print(f"  no={lst.no}, current={cur}   (* = 주목 노드)\n")

def make(*vals):
    """값들로 연결 리스트를 만든다"""
    l = LinkedList()
    for v in vals: l.add_last(v)
    return l

print("준비 완료 ✅\n")
draw(make('A','B','C','D','E','F'), "예시: A~F 연결 리스트")

---
# 🔁 [Remind] 워밍업 — 되감기

07장까지는 배열이 주인공이었어. 오늘부터 바뀌는 게 뭔지 먼저 잡자.

### R-1. 🟢 [설명] 배열의 강점과 약점

- **12일차 이진 검색**은 배열이어야만 가능했어. 왜지? (힌트: `a[mid]` 를 한 번에 꺼낼 수 있어야 함)
- **19~25일차 정렬**들도 전부 `a[i]`, `a[j]` 를 자유롭게 접근했지.
- 이걸 **임의 접근(random access)** 이라고 해. 배열의 최대 강점이야.
- 그럼 배열의 **약점**은? 교재 322p [그림 8-3]을 보고 한 문장으로: ①________________

### R-2. 🟡 [설명] 4일차 "call by object reference" 소환

```python
class Node:
    def __init__(self, data=None, next=None):
        self.data = data
        self.next = next
```

- `self.next = next` 에서 `next` 에 들어가는 건 **노드 자체**야, **노드에 대한 참조**야?
- 4일차에 배운 **"이름과 객체는 별개"** 를 떠올려봐. `a = b` 는 객체를 복사하지 않고 **같은 객체를 가리키게** 했지.
- 연결 리스트는 이 성질을 **정면으로 이용**하는 자료구조야. 노드를 옮기지 않고 **화살표만 바꿔서** 순서를 재배치하거든.
- 💡 그래서 오늘 나오는 모든 `=` 는 **"화살표를 다시 그린다"** 로 읽어야 해.

*(여기에 답 작성)*

---
# 🎯 PART 1 — 왜 연결 리스트인가 (1~3번)

### 1. 🟢 [설명] 배열 리스트의 문제

교재 322p [그림 8-3]. 회원 번호가 정렬된 배열에 **55번 회원**을 12와 33 사이에 삽입하려면?

```
삽입 전            삽입 후
0  12              0  12
1  33         →    1  55   ← 삽입
2  57              2  33
3  69              3  57
4  41              4  69
5  -               5  41
6  -               6  -
```

- 삽입 위치 뒤의 원소가 **몇 개** 밀려나야 해?
- 원소가 n개고 맨 앞에 삽입한다면 **최악 몇 개**가 밀릴까? 시간 복잡도는?
- 교재의 결론: **"데이터를 삽입·삭제함에 따라 데이터를 옮겨야 하므로 효율적이지 않습니다."**
- 🔥 **연결 리스트의 해법**: 데이터를 **옮기지 않고**, 대신 뭘 바꿀까? (교재 323p: "노드용 인스턴스를 생성하고, 데이터를 삭제할 때 노드용 인스턴스를 없애면")

*(답을 적은 뒤 실행)*

In [ ]:
import timeit
print("[배열 맨 앞 삽입의 비용]")
for n in (1000, 2000, 4000, 8000):
    a = list(range(n))
    t = timeit.timeit(lambda: a.insert(0, -1), number=200)
    del a[:200]
    print(f"  n={n:5d}: 200회 삽입에 {t*1000:6.2f}ms")
print("  → n이 2배면 시간도 약 2배 (한 번 삽입이 O(n))")

print("\n[연결 리스트 맨 앞 삽입]")
for n in (1000, 2000, 4000, 8000):
    l = make(*range(n))
    t = timeit.timeit(lambda: l.add_first(-1), number=200)
    print(f"  n={n:5d}: 200회 삽입에 {t*1000:6.2f}ms")
print("  → n이 커져도 시간이 그대로! (한 번 삽입이 O(1))")

### 2. 🟢 [설명] 노드와 자기 참조형

교재 323p [그림 8-4], [그림 8-5].

```python
class Node:
    def __init__(self, data=None, next=None):
        self.data = data      # 데이터
        self.next = next      # 뒤쪽 포인터
```

- 교재: **"Node는 데이터용 필드 data와는 별도로 자신과 같은 클래스형의 인스턴스를 참조하기 위한 참조용 필드 next를 갖습니다."**
- 이런 구조를 **①________형**(self-referential)이라고 해.
- 교재 323p: **"data는 데이터 자체가 아니라 '데이터에 대한 참조'이고 next는 '노드에 대한 참조'입니다."**
  → 그럼 `Node(5, None)` 을 만들면 메모리에 **몇 개의 객체**가 관련될까?
- **꼬리 노드의 `next` 값**은 뭐야? ②____  왜 그래야 하지?
- 🔥 만약 꼬리 노드의 `next` 가 다시 머리 노드를 가리킨다면? (08-4에서 배울 **원형 리스트**야)

*(답을 적은 뒤 실행)*

In [ ]:
n3 = Node('C', None)
n2 = Node('B', n3)
n1 = Node('A', n2)
print("직접 노드를 이어붙여 보기")
print(f"  n1.data = {n1.data}, n1.next.data = {n1.next.data}, n1.next.next.data = {n1.next.next.data}")
print(f"  n1.next.next.next = {n1.next.next.next}  ← 꼬리 노드의 next는 None\n")

print("[자기 참조형 확인]")
print(f"  type(n1)      = {type(n1).__name__}")
print(f"  type(n1.next) = {type(n1.next).__name__}   ← 자기와 같은 클래스!")

print("\n[data는 '데이터에 대한 참조'다]")
lst_data = [1, 2, 3]
node = Node(lst_data, None)
lst_data.append(4)
print(f"  Node에 리스트를 담고 원본을 수정하면: node.data = {node.data}")
print("  → 복사가 아니라 참조를 담고 있다 (4일차 call by object reference)")

### 3. 🟢 [설명] 용어 정리 — 이건 외워야 해

교재 321p [그림 8-2]. **딱 5개만** 정리하면 돼.

```
head → [A] → [B] → [C] → [D] → [E] → [F] → None
```

| 용어 | 영어 | 무엇 |
|---|---|---|
| ① 노드 | node | 리스트의 각 원소 |
| ② ____ 노드 | head node | 맨 앞의 노드 → 여기선 ____ |
| ③ ____ 노드 | tail node | 맨 끝의 노드 → 여기선 ____ |
| ④ ____ 노드 | predecessor node | 바로 앞에 있는 노드 |
| ⑤ ____ 노드 | successor node | 바로 뒤에 있는 노드 |

- 노드 `C`의 앞쪽 노드는? 뒤쪽 노드는?
- 교재 321p의 비유: **"마치 A가 B에게, B가 C에게 차례대로 연락하는 비상 연락망과 같은 구조입니다. 이런 구조에서는 누군가를 건너뛰거나 뒤돌아 앞 사람에게 연락해서는 안 됩니다."**
  → 이 비유가 말하는 연결 리스트의 **결정적 약점**은 뭐야? 🔥
  (`D`를 보려면 반드시 어디서부터 출발해야 하지?)

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C','D','E','F')
draw(l, "연결 리스트")

print("[C에서 앞쪽 노드로 갈 수 있을까?]")
ptr = l.head.next.next
print(f"  현재 노드: {ptr.data}")
print(f"  뒤쪽 노드: {ptr.next.data}   ← next 하나면 끝")
print(f"  앞쪽 노드: ???              ← 방법이 없다! 🔥")
print("\n  → 앞쪽으로 가려면 head부터 다시 출발해야 한다")
print("  → 이게 remove_last, remove 가 pre 변수를 쓰는 이유 (10~11번)")

print("\n[D에 접근하는 비용]")
steps = 0; ptr = l.head
while ptr.data != 'D':
    ptr = ptr.next; steps += 1
print(f"  head에서 D까지 {steps}칸 이동")
print(f"  배열이라면 a[3] 한 번에 접근 (O(1)) vs 연결 리스트는 O(n)")

---
# 🔍 PART 2 — 상태를 그림으로 판별하기 (4~6번)

> **여기가 "외우지 않기"의 핵심이야.** 판정식을 외우지 말고 **그림에서 읽어내는** 연습.

### 4. 🔴 [설명] 🔥 `head`는 머리 노드가 **아니다**

교재 326p가 콕 집어 경고해:
> **"head는 머리 노드에 대한 참조일 뿐 머리 노드 그 자체가 아님을 주의해야 합니다."**

이게 오늘 가장 헷갈리는 지점이야. 그림으로 구분해보자.

```
head [ ● ] ─────→ [A] → [B] → [C] → None
 ↑                 ↑
 이건 '상자'        이게 머리 노드
```

- `head` 는 `LinkedList` **객체의 필드**야. 리스트 밖에 있는 **하나의 변수**지.
- 머리 노드 `A` 는 **Node 객체**야.
- 그럼 빈 리스트일 때 `head` 는? ①____ (교재 326p: "참조해야 하는 노드가 존재하지 않으므로")

**질문 3개** 🔥
- `head.data` 는 뭘 뜻해? ②________________
- `head.next` 는? ③________________
- `head = head.next` 를 실행하면 무슨 일이 벌어져? **노드가 삭제될까?** ④____

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
print("head 자체와 머리 노드는 다른 것")
print(f"  type(l.head)  = {type(l.head).__name__}   ← head가 '참조하는' 것")
print(f"  l.head.data   = {l.head.data}          ← 머리 노드의 데이터")
print(f"  l.head.next.data = {l.head.next.data}       ← 2번째 노드의 데이터\n")

print("[head = head.next 를 하면?]")
old_head = l.head
l.head = l.head.next
draw(l, "  실행 후")
print(f"  원래 머리 노드 A는? old_head.data = {old_head.data}  ← 객체는 아직 살아 있다!")
print(f"  old_head.next.data = {old_head.next.data}  ← 여전히 B를 가리킨다")
print("\n  → 'head를 옮긴 것'이지 '노드를 삭제한 것'이 아니다.")
print("     다만 리스트에서 A를 가리키는 화살표가 없어져서,")
print("     아무도 참조하지 않으면 파이썬이 알아서 메모리를 회수한다 (13일차 GC 개념)")

### 5. 🟢 [그림] 판정식은 그림에서 나온다

교재 326~327p [그림 8-6, 8-7, 8-8]. **세 가지 그림을 직접 그리고**, 각각에서 판정식을 읽어내봐.

**(가) 빈 리스트** — 노드 0개
```
head → ①____
```
판정식: `head is ②____`

**(나) 노드 1개** — `A` 하나
```
head → [A] → ③____
```
- `head` 가 참조하는 곳: ④____
- `head.next` 가 참조하는 곳: ⑤____
- 판정식: `head.next is ⑥____`

**(다) 노드 2개** — `A`, `B`
```
head → [A] → [B] → ⑦____
```
- `head.next` 가 참조하는 곳: ⑧____
- `head.next.next` 가 참조하는 곳: ⑨____
- 판정식: `head.next.next is ⑩____`

**(라) 꼬리 노드 판별** — 어떤 노드 `p` 가 꼬리인지
- 판정식: `p.next is ⑪____`

🎯 **외우지 마.** `next` 를 따라가다 `None` 을 만나는 지점이 어디냐만 세면 돼.

- 교재 327p: **"지금까지 살펴본 3가지 경우의 판단은 no == 0, no == 1, no == 2를 사용할 수 있습니다."**
  → `no` 를 쓰는 방법과 `head` 를 따라가는 방법, **각각 장단점**은?

*(답을 적은 뒤 실행)*

In [ ]:
print("판정식 대조표\n")
for vals in ([], ['A'], ['A','B'], ['A','B','C']):
    l = make(*vals)
    h = l.head
    e0 = h is None
    e1 = (h is not None) and (h.next is None)
    e2 = (h is not None) and (h.next is not None) and (h.next.next is None)
    print(f"  {str(vals):18s} no={l.no}")
    print(f"    head is None          → {str(e0):5s}  (no==0 → {l.no==0})")
    print(f"    head.next is None     → {str(e1):5s}  (no==1 → {l.no==1})")
    print(f"    head.next.next is None→ {str(e2):5s}  (no==2 → {l.no==2})")
    print()

print("[꼬리 노드 판별]")
l = make('A','B','C')
ptr = l.head
while ptr is not None:
    print(f"  {ptr.data}: p.next is None → {ptr.next is None}  {'← 꼬리!' if ptr.next is None else ''}")
    ptr = ptr.next

### 6. 🟡 [손] `search()` 손으로 추적하기

교재 327~328p [그림 8-9, 8-10]. 노드 `D`를 검색하는 과정이야.

```
head → [A] → [B] → [C] → [D] → [E] → [F] → None
```

```python
cnt = 0
ptr = self.head
while ptr is not None:
    if ptr.data == data:
        self.current = ptr
        return cnt
    cnt += 1
    ptr = ptr.next
return -1
```

**표를 채워봐:**

| 단계 | ptr이 참조하는 노드 | cnt | `ptr.data == 'D'`? | 다음 동작 |
|---|---|---|---|---|
| 1 | A | 0 | ① | ② |
| 2 | ③ | ④ | ⑤ | ⑥ |
| 3 | | | | |
| 4 | | | | |

- 반환값은? ⑦____ 그리고 `current` 는 어느 노드를 가리키게 돼? ⑧____
- 교재 328p: **"cnt는 0부터 시작하는 값입니다(찾은 노드가 맨 앞이면 0입니다)."**
  → `linked_list_test.py` 에서는 `print(f'그 값의 데이터는 {pos + 1}번째에 있습니다')` 처럼 **+1** 을 해. 왜?
- 교재 328p가 정리한 **종료 조건 2가지**:
  - 종료 조건 1: ⑨________________
  - 종료 조건 2: ⑩________________
  → 이 둘이 코드의 **어느 줄**에 각각 해당해?
- 💡 이 알고리즘 어디서 봤지? **12일차 선형 검색**이야. 배열이 연결 리스트로 바뀌었을 뿐 구조가 같아.

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C','D','E','F')
target = 'D'
print(f"search('{target}') 추적\n")
cnt = 0; ptr = l.head; step = 0
while ptr is not None:
    step += 1
    hit = ptr.data == target
    print(f"  {step}단계: ptr → [{ptr.data}], cnt={cnt}, '{ptr.data}'=='{target}'? {hit}")
    if hit:
        print(f"         → current를 [{ptr.data}]로 설정하고 return {cnt}")
        break
    cnt += 1
    ptr = ptr.next
else:
    print("  → 꼬리까지 왔지만 못 찾음, return -1")

print()
print(f"실제 호출: l.search('{target}') = {l.search(target)}")
draw(l, "검색 후 상태")

print("실패하는 경우:")
print(f"  l.search('Z') = {l.search('Z')}   ← -1")
draw(l, "  실패 후 current는?")
print("  💡 검색에 실패하면 current는 바뀌지 않는다 (17번에서 다시)")

---
# ✏️ PART 3 — 삽입·삭제를 그림으로 (7~11번)

> **오늘 노트북의 핵심 파트.** 코드를 외우는 대신 **화살표를 그린다.**
>
> 🎯 **공통 규칙 하나만 기억해**:
> **"끊기 전에 먼저 이어라."** 화살표를 지우기 전에 새 화살표부터 그려야 노드를 잃어버리지 않아.

### 7. 🟢 [그림] `add_first` — 맨 앞에 삽입

교재 330p [그림 8-11]. 리스트 앞에 `G`를 넣어보자.

```
삽입 전:  head → [A] → [B] → [C] → None
```

```python
ptr = self.head                          # ①
self.head = self.current = Node(data, ptr)   # ②
self.no += 1
```

**직접 그려봐:**

**① `ptr = self.head` 실행 후** — 아직 아무것도 안 바뀜. `ptr` 이 가리키는 건? ①____

**② `Node(data, ptr)` 로 새 노드 G를 만들면** — G의 `next` 는 어디를 가리켜? ②____
```
        [G] → ③____
head → [A] → [B] → [C] → None
```

**③ `self.head = ...` 를 실행하면**
```
head → ④____
```
최종 그림:
```
head → [G] → [A] → [B] → [C] → None
```

**질문**
- `ptr` 변수가 **왜 필요할까?** `self.head = Node(data, self.head)` 로 한 줄에 쓸 수 있을까? 🔥
- 만약 순서를 바꿔서 `self.head = Node(data, None)` 먼저 하고 `self.head.next = ptr` 을 나중에 하면? 동작할까?
- **빈 리스트**에 `add_first` 를 하면? `ptr` 은 `None` 이 되고 새 노드의 `next` 도 `None` → 자연스럽게 처리돼. **따로 분기가 필요 없는 이유**를 설명해봐.

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
draw(l, "삽입 전")
old_head = l.head
l.add_first('G')
draw(l, "add_first('G') 후")
print(f"  새 머리 노드 G의 next → [{l.head.next.data}]  (= 삽입 전 머리 노드)")
print(f"  old_head는 여전히 [{old_head.data}] ← 기존 노드는 하나도 안 움직였다 🔥\n")

print("[한 줄로 써도 될까?]")
l2 = make('A','B','C')
l2.head = l2.current = Node('G', l2.head)   # ptr 없이
l2.no += 1
draw(l2, "  self.head = Node(data, self.head) 결과")
print("  → 된다! 파이썬은 오른쪽을 먼저 평가하므로 self.head가 '옛 값'으로 읽힌다")
print("  → 교재가 ptr을 쓴 건 '삽입 전 머리 노드'라는 의미를 이름으로 드러내려는 것\n")

print("[빈 리스트에 add_first]")
e = LinkedList()
e.add_first('G')
draw(e, "  결과")
print("  → ptr = None 이라 Node('G', None)이 되고, 이게 곧 꼬리 노드 조건 ✅")

### 8. 🟡 [그림] `add_last` — 맨 끝에 삽입

교재 331p [그림 8-12].

```python
if self.head is None:          # 리스트가 비어 있으면
    self.add_first(data)       # ← 재사용!
else:
    ptr = self.head
    while ptr.next is not None:    # ①
        ptr = ptr.next
    ptr.next = self.current = Node(data, None)   # ②
    self.no += 1
```

**직접 그려봐** — `head → [A] → [B] → [C] → None` 에 `G`를 붙이기

**① `while` 루프가 끝났을 때 `ptr` 이 가리키는 노드는?** ①____
- 교재 331p: **"while 문을 종료할 때 ptr은 꼬리 노드를 참조합니다."**
- 이 while문의 조건이 `ptr.next is not None` 인 이유를 5번의 꼬리 판정식과 연결해봐.

**② `ptr.next = Node(data, None)` 실행 후**
```
head → [A] → [B] → [C] → ②____ → None
```

**질문 3개** 🔥
- 새 노드의 `next` 를 **`None`으로** 만드는 이유는? (교재 331p: "맨 끝에 위치한 노드 G가 어떤 노드도 참조하지 않도록")
- `add_first` 는 리스트를 **훑지 않는데** `add_last` 는 **끝까지 훑어.** 각각의 시간 복잡도는? ③____ / ④____
- 빈 리스트일 때 `add_first` 를 **재사용**하는 이유는? 직접 처리하면 뭐가 문제야? (`ptr = self.head` 가 `None` 이면 `ptr.next` 에서 무슨 에러?)

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
draw(l, "삽입 전")
ptr = l.head; steps = 0
while ptr.next is not None:
    ptr = ptr.next; steps += 1
print(f"  꼬리 노드를 찾기까지 {steps}칸 이동 → ptr → [{ptr.data}]\n")
l.add_last('G')
draw(l, "add_last('G') 후")

print("[빈 리스트일 때 직접 처리하면?]")
e = LinkedList()
try:
    ptr = e.head          # None
    while ptr.next is not None:
        ptr = ptr.next
except AttributeError as ex:
    print(f"  AttributeError: {ex}")
    print("  → head가 None이면 ptr.next 자체가 불가능. 그래서 분기가 필요하다\n")

print("[add_first vs add_last 시간 복잡도 실측]")
for n in (500, 1000, 2000, 4000):
    l1 = make(*range(n)); l2 = make(*range(n))
    t = time.perf_counter()
    for _ in range(100): l1.add_first(-1)
    e1 = time.perf_counter() - t
    t = time.perf_counter()
    for _ in range(100): l2.add_last(-1)
    e2 = time.perf_counter() - t
    print(f"  n={n:5d}: add_first {e1*1000:6.2f}ms | add_last {e2*1000:7.2f}ms")
print("  → add_first는 n과 무관(O(1)), add_last는 n에 비례(O(n)) 🔥")

### 9. 🟢 [그림] `remove_first` — 머리 노드 삭제

교재 332p [그림 8-13].

```python
if self.head is not None:
    self.head = self.current = self.head.next
self.no -= 1
```

**직접 그려봐** — `head → [A] → [B] → [C] → None` 에서 `A`를 삭제

**실행 전**
```
head → [A] → [B] → [C] → None
```

**`self.head = self.head.next` 실행 후**
```
head → ①____ → [C] → None
        (A는 어디로?)
```

- 교재 332p: **"그 결과 삭제하기 전의 머리 노드 A는 어디에서도 참조되지 않습니다."**
  → **"삭제"** 라는 게 정확히 무슨 일이야? `del` 을 쓴 것도 아닌데. ②________________
- 노드가 **1개뿐**일 때 `remove_first` 를 하면? `head.next` 는 `None` 이니 `head = None` 이 되어 **빈 리스트**가 돼. 별도 분기가 필요 없지?
- 🔥 **이 코드에는 버그가 하나 있어.** `self.no -= 1` 의 **들여쓰기**를 잘 봐. 20번에서 다룰 거야.

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
draw(l, "삭제 전")
old = l.head
l.remove_first()
draw(l, "remove_first() 후")
print(f"  old(=A 노드)는 아직 객체로 살아 있다: old.data = {old.data}")
print(f"  하지만 리스트에서 old를 가리키는 화살표가 없다 → 참조 카운트 0이 되면 자동 회수\n")

print("[노드 1개일 때]")
one = make('A')
draw(one, "  삭제 전")
one.remove_first()
draw(one, "  삭제 후")
print("  → head.next가 None이라 head = None → 빈 리스트 ✅ (분기 불필요)")

### 10. 🔴 [그림] `remove_last` — 꼬리 노드 삭제 (두 커서!)

교재 333p [그림 8-14]. **여기서 `pre` 라는 두 번째 커서가 등장해.**

```python
ptr = self.head       # 스캔 중인 노드
pre = self.head       # 스캔 중인 노드의 앞쪽 노드
while ptr.next is not None:
    pre = ptr         # ①
    ptr = ptr.next    # ②
pre.next = None       # ③
self.current = pre
self.no -= 1
```

**🔥 왜 커서가 두 개나 필요할까?**
- 꼬리 노드 `F`를 삭제하려면, `F`를 가리키던 화살표를 끊어야 해.
- 그 화살표는 **누구의** `next` 야? ①____
- 그런데 3번에서 봤듯 연결 리스트는 **앞쪽으로 갈 수 없어.** `ptr` 이 `F`에 도착한 뒤에는 `E`로 되돌아갈 방법이 없지.
- 그래서 **한 칸 뒤처져 따라오는 커서** `pre` 가 필요해.

**직접 그려봐** — `head → [A] → [B] → [C] → None`

| 반복 | pre | ptr | `ptr.next is not None`? |
|---|---|---|---|
| 시작 | A | A | ② |
| 1회 후 | ③ | ④ | ⑤ |
| 2회 후 | ⑥ | ⑦ | ⑧ (종료) |

- 루프 종료 시 `ptr` 은 ⑨____, `pre` 는 ⑩____ 를 가리켜.
- 교재 334p: **"while 문을 종료할 때 pre가 참조하는 곳은 노드 E이고, ptr이 참조하는 곳은 노드 F입니다."**
- `pre.next = None` 을 실행하면 무슨 일이? ⑪________________

**질문**: 18일차 셰이커 정렬, 21일차 퀵 정렬, 23일차 병합의 커서들과 `pre`/`ptr` 의 **차이**는 뭘까? (힌트: 그건 인덱스, 이건?)

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C')
draw(l, "삭제 전")
ptr = l.head; pre = l.head; step = 0
print(f"  시작: pre → [{pre.data}], ptr → [{ptr.data}]")
while ptr.next is not None:
    step += 1
    pre = ptr
    ptr = ptr.next
    print(f"  {step}회 후: pre → [{pre.data}], ptr → [{ptr.data}], "
          f"ptr.next is not None? {ptr.next is not None}")
print(f"\n  종료: ptr = 꼬리 [{ptr.data}], pre = 맨 끝에서 2번째 [{pre.data}]")
l.remove_last()
draw(l, "remove_last() 후")

print("[노드 1개일 때는 remove_first를 재사용]")
one = make('A')
one.remove_last()
draw(one, "  결과")
print("  → head.next is None 이면 pre/ptr을 쓸 수 없으니 분기 필요 ✅")

### 11. 🔴 [그림] `remove(p)` — 임의의 노드 삭제

교재 335p [그림 8-15]. 노드 `D`를 삭제해보자.

```python
if p is self.head:
    self.remove_first()
else:
    ptr = self.head
    while ptr.next is not p:      # ①
        ptr = ptr.next
        if ptr is None:
            return                # ② p가 리스트에 없음
    ptr.next = p.next             # ③
    self.current = ptr
    self.no -= 1
```

**직접 그려봐** — `head → [A] → [B] → [C] → [D] → [E] → None` 에서 `D` 삭제

**① `while ptr.next is not p` 가 끝났을 때 `ptr` 은?** ①____
- 교재 335p: **"ptr.next가 p와 같아지면 while 문은 종료합니다. 이때 ptr이 참조하는 곳은 삭제할 노드 D의 앞쪽 노드인 C가 됩니다."**

**② `ptr.next = p.next` 실행 후**
```
head → [A] → [B] → [C] ──────→ ②____ → None
                          [D]  ← 아무도 참조 안 함
```

**질문 4개**
- `p is self.head` 를 **먼저 검사**하는 이유는? 안 하면 어떤 문제가? 🔥
  (머리 노드에는 **앞쪽 노드가 없잖아**)
- `is` 를 쓰고 `==` 를 안 쓴 이유는? (3일차 `is` vs `==`)
- `if ptr is None: return` 이 없으면 무슨 에러가 날까?
- 10번의 `remove_last` 와 비교해봐. 둘 다 "앞쪽 노드를 찾는" 작업인데 **방법이 달라.**
  - `remove_last`: `pre`/`ptr` **두 커서**로 따라감
  - `remove`: `ptr.next` 를 **미리 내다봄**
  → 같은 문제를 푸는 두 가지 방식이야. 각각의 장단점은?

*(답을 적은 뒤 실행)*

In [ ]:
l = make('A','B','C','D','E')
draw(l, "삭제 전")
p = l.head.next.next.next        # D 노드
print(f"  삭제 대상 p → [{p.data}]")
ptr = l.head; step = 0
while ptr.next is not p:
    ptr = ptr.next; step += 1
print(f"  {step}칸 이동 후 ptr → [{ptr.data}]  (= p의 앞쪽 노드)\n")
l.remove(p)
draw(l, "remove(D) 후")
print(f"  삭제된 D의 next는? [{p.next.data}]  ← 노드 자체는 여전히 E를 가리킨다")
print("  하지만 리스트에서 D를 가리키는 화살표가 사라졌다\n")

print("[머리 노드를 remove로 지우면?]")
l2 = make('A','B','C')
l2.remove(l2.head)
draw(l2, "  remove(head) 결과")
print("  → p is self.head 분기가 remove_first로 넘겨줬다 ✅")

print("[리스트에 없는 노드를 지우려 하면?]")
l3 = make('A','B','C')
outsider = Node('Z', None)
l3.remove(outsider)
draw(l3, "  remove(외부 노드) 결과")
print("  → if ptr is None: return 이 조용히 막아준다 (변화 없음)")

---
# 💻 PART 4 — 코드 구현 (12~16번)

> 7~11번에서 그림으로 유도한 걸 이제 코드로. **그림이 그려지면 코드는 저절로 나와.**

### 12. 🟢 [빈칸] `Node` 와 `LinkedList` 뼈대

**기대 출력**
```
Node: data=A, next=None
빈 리스트: head=None, current=None, no=0
len() = 0
```

In [ ]:
class MyNode:
    """연결 리스트용 노드 클래스"""
    def __init__(self, data: Any = None, next: MyNode = None):
        self.data = ___              # ① 데이터
        self.next = ___              # ② 뒤쪽 포인터


class MyLinkedList:
    """연결 리스트 클래스"""
    def __init__(self) -> None:
        self.no = ___                # ③ 노드의 개수
        self.head = ___              # ④ 머리 노드에 대한 참조
        self.current = ___           # ⑤ 주목 노드에 대한 참조

    def __len__(self) -> int:
        return ___                   # ⑥


n = MyNode('A')
print(f"Node: data={n.data}, next={n.next}")
l = MyLinkedList()
print(f"빈 리스트: head={l.head}, current={l.current}, no={l.no}")
print(f"len() = {len(l)}")

### 13. 🟡 [빈칸] `search` 와 `__contains__`

6번에서 추적한 그대로.

**기대 출력**
```
search('D') = 3, current = D
search('Z') = -1
'C' in lst = True
'Z' in lst = False
```

In [ ]:
def my_search(self, data: Any) -> int:
    """data와 값이 같은 노드를 검색"""
    cnt = 0
    ptr = ___                        # ① 어디서 출발?
    while ___:                       # ② 종료 조건 1 (꼬리를 지나쳤나)
        if ___:                      # ③ 종료 조건 2 (찾았나)
            self.current = ___       # ④ 주목 노드 갱신
            return ___               # ⑤
        cnt += 1
        ptr = ___                    # ⑥ 한 칸 뒤로
    return ___                       # ⑦ 못 찾았을 때

def my_contains(self, data: Any) -> bool:
    return ___                       # ⑧ search를 재사용

MyLinkedList.search = my_search
MyLinkedList.__contains__ = my_contains
MyLinkedList.add_last = LinkedList.add_last
MyLinkedList.add_first = LinkedList.add_first

lst = MyLinkedList()
for v in ['A','B','C','D']: lst.add_last(v)
print(f"search('D') = {lst.search('D')}, current = {lst.current.data}")
print(f"search('Z') = {lst.search('Z')}")
print(f"'C' in lst = {'C' in lst}")
print(f"'Z' in lst = {'Z' in lst}")

### 14. 🟡 [빈칸] `add_first` 와 `add_last`

7~8번의 그림 그대로.

**기대 출력**
```
add_first: ['X', 'A', 'B', 'C'] no=4
add_last : ['X', 'A', 'B', 'C', 'Y'] no=5
빈 리스트에 add_last: ['Z'] no=1
```

In [ ]:
def my_add_first(self, data: Any) -> None:
    """맨 앞에 노드를 삽입"""
    ptr = ___                                # ① 삽입 전의 머리 노드를 기억
    self.head = self.current = Node(data, ___)   # ② 새 노드의 next는?
    self.no += 1


def my_add_last(self, data: Any) -> None:
    """맨 끝에 노드를 삽입"""
    if ___:                                  # ③ 리스트가 비어 있으면
        self.___(data)                       # ④ 어느 함수를 재사용?
    else:
        ptr = self.head
        while ___:                           # ⑤ 꼬리 노드를 찾을 때까지
            ptr = ptr.next
        ptr.next = self.current = Node(data, ___)   # ⑥ 새 꼬리의 next는?
        self.no += 1


MyLinkedList.add_first = my_add_first
MyLinkedList.add_last = my_add_last

def vals(l):
    r = []; p = l.head
    while p: r.append(p.data); p = p.next
    return r

lst = MyLinkedList()
for v in ['A','B','C']: lst.add_last(v)
lst.add_first('X'); print(f"add_first: {vals(lst)} no={lst.no}")
lst.add_last('Y');  print(f"add_last : {vals(lst)} no={lst.no}")
e = MyLinkedList(); e.add_last('Z')
print(f"빈 리스트에 add_last: {vals(e)} no={e.no}")

### 15. 🔴 [빈칸] `remove_first`, `remove_last`, `remove`

9~11번의 그림 그대로. **가장 어려운 문제야.**

**기대 출력**
```
remove_first: ['B', 'C', 'D'] no=3
remove_last : ['B', 'C'] no=2
remove(C)   : ['B'] no=1
빈 리스트 안전: [] no=0
```

In [ ]:
def my_remove_first(self) -> None:
    """머리 노드를 삭제"""
    if ___:                          # ① 리스트가 비어 있지 않으면
        self.head = self.current = ___   # ② 머리를 어디로?
        self.no -= 1                 # (20번에서 다룰 위치! 여기가 맞다)


def my_remove_last(self) -> None:
    """꼬리 노드를 삭제"""
    if self.head is not None:
        if ___:                      # ③ 노드가 1개뿐이면
            self.remove_first()
        else:
            ptr = self.head          # 스캔 중인 노드
            pre = self.head          # 스캔 중인 노드의 앞쪽 노드
            while ___:               # ④ 꼬리를 찾을 때까지
                pre = ___            # ⑤ 한 칸 뒤처져 따라간다
                ptr = ___            # ⑥
            ___ = None               # ⑦ 꼬리로의 화살표를 끊는다
            self.current = pre
            self.no -= 1


def my_remove(self, p: Node) -> None:
    """노드 p를 삭제"""
    if self.head is not None:
        if ___:                      # ⑧ p가 머리 노드이면 (== 아님 주의!)
            self.remove_first()
        else:
            ptr = self.head
            while ___:               # ⑨ p의 앞쪽 노드를 찾을 때까지
                ptr = ptr.next
                if ptr is None:
                    return           # p가 리스트에 없음
            ptr.next = ___           # ⑩ 앞쪽 노드가 p의 뒤쪽을 가리키게
            self.current = ptr
            self.no -= 1


MyLinkedList.remove_first = my_remove_first
MyLinkedList.remove_last = my_remove_last
MyLinkedList.remove = my_remove

lst = MyLinkedList()
for v in ['A','B','C','D']: lst.add_last(v)
lst.remove_first(); print(f"remove_first: {vals(lst)} no={lst.no}")
lst.remove_last();  print(f"remove_last : {vals(lst)} no={lst.no}")
lst.remove(lst.head.next); print(f"remove(C)   : {vals(lst)} no={lst.no}")
e = MyLinkedList(); e.remove_first(); e.remove_last()
print(f"빈 리스트 안전: {vals(e)} no={e.no}")

### 16. 🟡 [빈칸] `clear`, `next`, 출력 함수들

**기대 출력**
```
next() 3번: A → B → C → D, 4번째는 False
clear 후: [] no=0 current=None
```

In [ ]:
def my_clear(self) -> None:
    """전체 노드를 삭제"""
    while ___:                       # ① 리스트가 빌 때까지
        self.___()                   # ② 어느 함수를 반복?
    self.current = ___               # ③
    self.no = ___                    # ④


def my_next(self) -> bool:
    """주목 노드를 한 칸 뒤로 이동"""
    if ___ or ___:                   # ⑤⑥ 이동할 수 없는 두 경우
        return False
    self.current = ___               # ⑦
    return True


MyLinkedList.clear = my_clear
MyLinkedList.next = my_next

lst = MyLinkedList()
for v in ['A','B','C','D']: lst.add_last(v)
lst.search('A')
path = [lst.current.data]
while lst.next():
    path.append(lst.current.data)
print(f"next() 3번: {' → '.join(path)}, 4번째는 {lst.next()}")
lst.clear()
print(f"clear 후: {vals(lst)} no={lst.no} current={lst.current}")

---
# 🎯 PART 5 — `current` 와 이터레이터 (17~19번)

> **"외울 게 많다"의 정점인 표 8-1을, 원리 하나로 무너뜨린다.**

### 17. 🔴 [설명] 🔥🔥 표 8-1을 외우지 마라

교재 338p [표 8-1]은 각 함수 실행 후 `current` 값을 정리한 표야. **11줄이나 돼.**

| 실행한 함수 | current의 값 |
|---|---|
| `__init__()` | None |
| `search()` | 검색에 성공하면 발견한 노드 |
| `add_first()` | 삽입한 머리 노드 |
| `add_last()` | 삽입한 꼬리 노드 |
| `remove_first()` | 삭제한 뒤 머리 노드 |
| `remove_last()` | 삭제한 뒤 꼬리 노드 |
| `remove()` | 삭제한 노드의 앞쪽 노드 |
| `remove_current_node()` | 삭제한 노드의 앞쪽 노드 |
| `clear()` | None |
| `next()` | 이동한 뒤 주목 노드 |
| `print_current_node()`, `print()` | 업데이트하지 않음 |

**이걸 통째로 외우려 하지 마.** 🎯 아래 **한 문장**으로 전부 유도돼:

> ### 💡 **"그 함수가 마지막으로 손댄, 리스트에 아직 남아 있는 노드"**

**직접 검증해봐.** 각 함수에 대해 이 원리를 적용하면 표의 값이 나오는지:

| 함수 | 마지막으로 손댄 노드는? | 그게 리스트에 남아 있나? | → current |
|---|---|---|---|
| `add_first()` | ① | ② | 삽입한 머리 노드 ✅ |
| `add_last()` | ③ | ④ | ⑤ |
| `remove_first()` | 삭제한 머리 노드 | ❌ 없어졌다 | → 그럼 그 **다음**인 ⑥ |
| `remove_last()` | 삭제한 꼬리 노드 | ❌ | → 그럼 그 **앞**인 ⑦ |
| `remove(p)` | 삭제한 p | ❌ | → 그럼 그 **앞**인 ⑧ |
| `clear()` | 전부 삭제 | ❌ 아무것도 없음 | ⑨ |
| `next()` | ⑩ | ✅ | ⑪ |
| `print()` | ⑫ 리스트를 **바꾸지 않음** | - | ⑬ |

**추가 질문 2개**
- `search()` 가 **실패**하면 `current` 는 어떻게 돼? 왜? (코드의 `self.current = ptr` 이 어디에 있는지 봐) 🔥
- `print()` 와 `print_current_node()` 가 `current` 를 안 건드리는 이유를 원리로 설명하면? ⑭________________

*(표를 채운 뒤 실행해서 대조)*

In [ ]:
def fresh():
    l = LinkedList()
    for v in ['A','B','C','D']: l.add_last(v)
    l.search('A')          # current를 A로 통일
    return l

tests = [
    ("search('C')",        lambda l: l.search('C')),
    ("search('Z') 실패",    lambda l: l.search('Z')),
    ("add_first('X')",     lambda l: l.add_first('X')),
    ("add_last('Y')",      lambda l: l.add_last('Y')),
    ("remove_first()",     lambda l: l.remove_first()),
    ("remove_last()",      lambda l: l.remove_last()),
    ("remove(B노드)",       lambda l: l.remove(l.head.next)),
    ("clear()",            lambda l: l.clear()),
    ("next() (B에서)",      lambda l: (l.search('B'), l.next())),
    ("print()",            lambda l: l.print()),
    ("for e in lst",       lambda l: [e for e in l]),
]

print("함수 실행 전 current는 항상 A\n")
print(f"{'실행한 함수':22s} | current | 원리로 설명")
print("-" * 72)
reasons = {
    "search('C')": "찾은 노드를 손댔고 남아 있음",
    "search('Z') 실패": "아무 노드도 안 손댐 → 그대로",
    "add_first('X')": "삽입한 노드를 손댔고 남아 있음",
    "add_last('Y')": "삽입한 노드를 손댔고 남아 있음",
    "remove_first()": "머리를 지웠으니 남은 건 그 다음",
    "remove_last()": "꼬리를 지웠으니 남은 건 그 앞(pre)",
    "remove(B노드)": "B를 지웠으니 남은 건 그 앞(ptr)",
    "clear()": "남은 노드가 없음",
    "next() (B에서)": "이동한 노드를 손댐",
    "print()": "리스트를 안 바꿈 → 그대로",
    "for e in lst": "이터레이터는 별도 커서를 씀 (18번)",
}
for name, fn in tests:
    l = fresh()
    with contextlib.redirect_stdout(io.StringIO()):
        fn(l)
    after = l.current.data if l.current else None
    print(f"{name:22s} |   {str(after):5s} | {reasons[name]}")

print("\n🎯 11줄을 외운 게 아니라, 한 문장으로 전부 유도됐다.")

### 18. 🟡 [빈칸] 이터레이터 구현

교재 338p 보충수업 8-2:
> **"이터러블 객체는 원소를 1개씩 꺼내는 구조의 객체입니다. ... 이터레이터의 `__next__()` 함수를 호출하거나, 내장 함수인 `next()` 함수에 반복자를 전달하면 줄지어 늘어선 원소를 순차적으로 꺼냅니다. 꺼낼 원소가 없으면 StopIteration 예외 처리를 내보냅니다."**

**두 클래스의 역할 분담**

| | 구현하는 함수 | 역할 |
|---|---|---|
| `LinkedList` | `__iter__` | ① |
| `LinkedListIterator` | `__iter__`, `__next__` | ② |

- 🔥 **왜 이터레이터를 별도 클래스로 분리할까?** `LinkedList` 자신이 `__next__` 를 가지면 안 될까?
  💡 힌트: 같은 리스트를 **중첩 for문**으로 두 번 동시에 돌리면?
- `LinkedListIterator` 에도 `current` 필드가 있어. 이건 `LinkedList.current` 와 **같은 거야?** ③____
  → 교재 339p: **"실습 8-1 프로그램의 이터레이터는 주목 포인터 current를 업데이트하지 않습니다."**

**기대 출력**
```
for 순회: A B C D
next() 직접: A B C
StopIteration 확인 ✅
중첩 for: 16개 쌍
```

In [ ]:
class MyIterator:
    """LinkedList의 이터레이터용 클래스"""
    def __init__(self, head: Node):
        self.current = ___                  # ① 어디서 시작?

    def __iter__(self) -> 'MyIterator':
        return ___                          # ② 자기 자신

    def __next__(self) -> Any:
        if ___:                             # ③ 더 꺼낼 게 없으면
            raise ___                       # ④ 어떤 예외?
        else:
            data = ___                      # ⑤ 지금 노드의 데이터
            self.current = ___              # ⑥ 한 칸 뒤로
            return data


def my_iter(self) -> MyIterator:
    return MyIterator(___)                  # ⑦ 무엇을 넘겨줄까?

MyLinkedList.__iter__ = my_iter

lst = MyLinkedList()
for v in ['A','B','C','D']: lst.add_last(v)

print("for 순회:", *[e for e in lst])

it = iter(lst)
print("next() 직접:", next(it), next(it), next(it))
next(it)
try:
    next(it)
except StopIteration:
    print("StopIteration 확인 ✅")

pairs = [(a, b) for a in lst for b in lst]
print(f"중첩 for: {len(pairs)}개 쌍")

### 19. 🟡 [실험] 이터레이터는 `current` 를 안 건드린다

17번 표의 마지막 줄과 이어지는 이야기야.

- `for e in lst:` 를 돌리면 `LinkedList.current` 는 바뀔까? **예측**해봐.
- 왜 그렇게 설계했을까? 🔥
  💡 만약 이터레이터가 `LinkedList.current` 를 갱신한다면, 순회가 끝난 뒤 `current` 는 어디를 가리키게 될까? 사용자가 원하던 주목 노드가 날아가지 않을까?
- **`search()` 후에 `for` 순회를 하고, 다시 `print_current_node()` 를 하면** 뭐가 출력될까?

*(예측을 적은 뒤 실행)*

In [ ]:
lst = make('A','B','C','D')
lst.search('C')
print(f"search('C') 후 current = {lst.current.data}")

for e in lst:
    pass
print(f"for 순회 후    current = {lst.current.data}   ← 그대로! ✅")

lst.print_current_node()

print("\n[만약 이터레이터가 current를 갱신한다면?]")
class BadIterator:
    def __init__(self, lst):
        self.lst = lst
        self.node = lst.head
    def __iter__(self): return self
    def __next__(self):
        if self.node is None: raise StopIteration
        self.lst.current = self.node          # 🐛 원본의 current를 건드린다
        d = self.node.data
        self.node = self.node.next
        return d

l2 = make('A','B','C','D')
l2.search('C')
print(f"  search('C') 후 current = {l2.current.data}")
for e in BadIterator(l2): pass
print(f"  나쁜 이터레이터 순회 후 current = {l2.current.data}  ← 꼬리로 끌려갔다 ❌")
print("\n  → 순회는 '읽기' 작업인데 상태를 바꾸면 부작용(side effect)이 된다")

---
# 🐛 PART 6 — 버그와 성능 (20~22번)

### 20. 🔴 [디버깅] 🔥🔥 교재 코드의 진짜 버그

9번에서 예고했던 거야. `remove_first` 를 다시 봐:

```python
def remove_first(self) -> None:
    if self.head is not None:
        self.head = self.current = self.head.next
    self.no -= 1                    # ← 들여쓰기!
```

`self.no -= 1` 이 **`if` 블록 바깥**에 있어.

- **빈 리스트**에서 `remove_first()` 를 호출하면 무슨 일이? ①________________
- 그러면 `len(lst)` 는 어떻게 될까? **예측**해봐. 그냥 음수가 나올까, 에러가 날까? 🔥
- `remove_last` 는 어떻게 되어 있어? 두 함수를 나란히 놓고 **비대칭**을 확인해봐.
- 이 버그를 고치려면 몇 글자를 바꿔야 해?

**연결 지점**
- 18일차 recurring 버그: **들여쓰기가 만든 조용한 오류**
- 22일차 14번: 교재 코드 자체의 잠재 버그(`while j > 0`)
- 28일차 12번: 교재 코드의 `pt` 초기화 누락
- → **교재 코드도 완벽하지 않다.** 읽을 때 항상 의심하는 습관이 필요해.

*(예측을 적은 뒤 실행)*

In [ ]:
print("[빈 리스트에서 remove_first]")
e = LinkedList()
print(f"  초기: no = {e.no}")
e.remove_first()
print(f"  1회 후: no = {e.no}   ← 음수! 🔥")
e.remove_first(); e.remove_first()
print(f"  3회 후: no = {e.no}")

print("\n[그래서 len()이 터진다]")
try:
    print(len(e))
except ValueError as ex:
    print(f"  ValueError: {ex}")
    print("  → 파이썬은 __len__이 음수를 반환하면 예외를 던진다")

print("\n[remove_last와 비교 — 비대칭]")
e2 = LinkedList()
e2.remove_last()
print(f"  빈 리스트에 remove_last 후: no = {e2.no}   ← 이건 정상!")
print("\n  remove_last 코드:")
print("    if self.head is not None:")
print("        ...")
print("            self.no -= 1        ← if 안쪽 ✅")
print("  remove_first 코드:")
print("    if self.head is not None:")
print("        ...")
print("    self.no -= 1                ← if 바깥 ❌")

print("\n[고친 버전]")
class FixedLL(LinkedList):
    def remove_first(self) -> None:
        if self.head is not None:
            self.head = self.current = self.head.next
            self.no -= 1              # ✅ 들여쓰기 한 단계만 넣으면 끝
f = FixedLL()
f.remove_first(); f.remove_first()
print(f"  빈 리스트에 2회 호출 후: no = {f.no}, len = {len(f)} ✅")

### 21. 🟡 [실험] `add_last` 는 왜 느린가

8번에서 `add_last` 가 O(n)이라는 걸 봤지. 그럼 **n개를 순서대로 넣으면** 총 비용은?

- 1번째 삽입: 리스트가 비어 있으니 `add_first` 로 넘어가 스캔 ①____ 칸
- 2번째 삽입: 스캔 ②____ 칸 (노드가 1개뿐)
- 3번째 삽입: 스캔 ____ 칸
- ...
- n번째 삽입: 스캔 ③____ 칸
- **총합 = 0 + 0 + 1 + 2 + ... + (n-2) = ④________**

- 그럼 n개를 `add_last` 로 만드는 비용은 **O(?)** ⑤____
- 🔥 이걸 어떻게 고칠 수 있을까? (힌트: 꼬리 노드를 매번 찾지 말고 **기억해두면**?)
- 💡 실제 자료구조 라이브러리는 `tail` 포인터를 함께 유지해서 `add_last` 를 O(1)로 만들어. 08-4의 **원형 이중 연결 리스트**가 그 방향이야.

*(계산한 뒤 실행)*

In [ ]:
class CountLL(LinkedList):
    scans = 0
    def add_last(self, data):
        if self.head is None:
            self.add_first(data)
        else:
            ptr = self.head
            while ptr.next is not None:
                ptr = ptr.next; CountLL.scans += 1
            ptr.next = self.current = Node(data, None); self.no += 1

print("n개를 add_last로 넣을 때 총 스캔 횟수\n")
print("     n | 총 스캔    | (n-1)(n-2)/2")
print("-" * 36)
for n in (100, 200, 400, 800):
    CountLL.scans = 0
    l = CountLL()
    for i in range(n): l.add_last(i)
    print(f"  {n:4d} | {CountLL.scans:8,d} | {(n-1)*(n-2)//2:12,d}")
print("\n→ 정확히 (n-1)(n-2)/2 ≈ n²/2 = O(n²) 🔥")

print("\n[tail 포인터를 유지하면?]")
class TailLL(LinkedList):
    def __init__(self):
        super().__init__(); self.tail = None
    def add_last(self, data):
        node = Node(data, None)
        if self.head is None:
            self.head = node
        else:
            self.tail.next = node
        self.tail = self.current = node
        self.no += 1

for n in (2000, 4000, 8000):
    t = time.perf_counter()
    l1 = LinkedList()
    for i in range(n): l1.add_last(i)
    e1 = time.perf_counter() - t
    t = time.perf_counter()
    l2 = TailLL()
    for i in range(n): l2.add_last(i)
    e2 = time.perf_counter() - t
    print(f"  n={n:5d}: 교재판 {e1:6.3f}s | tail 유지 {e2:.4f}s  ({e1/e2:,.0f}배)")

### 22. 🟢 [정리] 배열 vs 연결 리스트

교재 325p 보충수업 8-1:
> **"연결 리스트는 임의의 위치에 원소를 삽입하거나 삭제할 때 빠르게 수행할 수 있다는 장점이 있습니다. 하지만 기억 영역(메모리)과 속도 면에서는 배열보다 효율이 뒤떨어집니다."**
> **"파이썬의 리스트는 이러한 연결 리스트의 자료구조가 아니라 모든 원소를 연속으로 메모리에 배치하는 '배열'로 내부에서 구현하고 있습니다."**

**표를 완성해봐:**

| 연산 | 배열 (파이썬 list) | 연결 리스트 |
|---|---|---|
| i번째 원소 접근 | O(1) | ① |
| 맨 앞 삽입 | ② | ③ |
| 맨 끝 삽입 | O(1) (평균) | ④ (교재판) / ⑤ (tail 유지) |
| 맨 앞 삭제 | ⑥ | ⑦ |
| 검색 | O(n) | ⑧ |
| 노드 1개당 메모리 | 포인터 1개 | ⑨ 데이터 + 포인터 + 객체 오버헤드 |

**최종 질문 3개**
1. 연결 리스트가 배열보다 확실히 유리한 상황은? 한 가지만 정확히.
2. 교재 325p: **"원소를 하나씩 추가·삽입할 때마다 내부에서 메모리를 확보하거나 해제하지 않습니다. 실제 필요한 메모리보다 여유 있게 미리 마련해 놓기 때문입니다."**
   → 파이썬 `list.append()` 가 평균 O(1)인 이유가 이거야. 그럼 **최악**에는 어떻게 될까?
3. 08-3에서는 **커서를 이용한 연결 리스트**(배열 안에 노드를 저장)를 배워. 교재 342p가 그 동기를 이렇게 말해:
   **"노드를 삽입·삭제할 때마다 내부에서 노드용 인스턴스를 생성하고 소멸합니다. 이때 메모리를 확보하고 해제하는 데 쓰는 비용을 결코 무시할 수 없습니다."**
   → 이게 오늘 배운 연결 리스트의 **어떤 약점**을 겨냥한 걸까?

*(답을 적은 뒤 실행)*

In [ ]:
print("[메모리 비교]")
n = 1000
l = make(*range(n))
node = l.head
node_size = sys.getsizeof(node) + sys.getsizeof(node.__dict__)
print(f"  Node 1개 ≈ {node_size} 바이트 (객체 헤더 + __dict__)")
print(f"  노드 {n}개 ≈ {node_size*n/1024:.1f} KB")
pl = list(range(n))
print(f"  파이썬 list {n}개 = {sys.getsizeof(pl)/1024:.1f} KB (포인터 배열만)")
print(f"  → 약 {node_size*n/sys.getsizeof(pl):.0f}배 차이 🔥\n")

print("[연산별 속도 비교 — n=3000]")
n = 3000
results = []

a = list(range(n))
t = time.perf_counter()
for _ in range(300): a.insert(0, -1)
results.append(("맨 앞 삽입", "배열", time.perf_counter()-t))
del a[:300]

l = make(*range(n))
t = time.perf_counter()
for _ in range(300): l.add_first(-1)
results.append(("맨 앞 삽입", "연결", time.perf_counter()-t))

a = list(range(n))
t = time.perf_counter()
for i in range(300): _ = a[i]
results.append(("i번째 접근", "배열", time.perf_counter()-t))

l = make(*range(n))
t = time.perf_counter()
for i in range(300):
    p = l.head
    for _ in range(i): p = p.next
results.append(("i번째 접근", "연결", time.perf_counter()-t))

for op in ("맨 앞 삽입", "i번째 접근"):
    rows = [r for r in results if r[0] == op]
    print(f"  {op}: 배열 {rows[0][2]*1000:7.2f}ms | 연결 {rows[1][2]*1000:7.2f}ms  "
          f"→ {'연결 승 ✅' if rows[1][2] < rows[0][2] else '배열 승 ✅'}")

---
---

# ✅ 정답 & 해설

> ⚠️ **화살표를 직접 그린 뒤에 내려와.** 특히 7~11번은 손으로 그려야 남아.

---

## 🔁 Remind

### R-1
- 이진 검색은 `a[mid]` 를 **한 번에** 꺼낼 수 있어야 가능해. 연결 리스트로는 mid에 가려면 mid칸을 걸어야 하니 이진 검색의 이점이 사라져.
- ① **삽입·삭제할 때 뒤의 원소를 전부 밀어야 한다 (O(n))**

### R-2
- `next` 에 들어가는 건 **노드에 대한 참조**야. 노드 객체를 복사하는 게 아니지.
- 연결 리스트는 **노드를 옮기지 않고 화살표만 바꿔서** 순서를 재배치해. 그래서 삽입/삭제가 O(1)이 될 수 있어.
- 💡 오늘의 모든 `=` 는 **"화살표를 다시 그린다"** 로 읽어야 해. 이 습관 하나면 7~11번이 전부 풀려.

---

## 🎯 PART 1 해설

### 1. 배열 리스트의 문제
- 55를 인덱스 1에 넣으면 **뒤의 4개**(33, 57, 69, 41)가 전부 한 칸씩 밀려.
- 맨 앞에 삽입하면 **n개 전부** → **O(n)**
- 🔥 연결 리스트의 해법: 데이터를 옮기는 대신 **화살표(포인터)만 바꾼다.**

**실측**: 배열은 n이 2배면 시간도 2배(O(n)), 연결 리스트 `add_first` 는 n이 커져도 **시간이 그대로**(O(1)).

---

### 2. 노드와 자기 참조형
- ① **자기 참조**(self-referential)형
- `Node(5, None)` 은 Node 객체 1개 + 그 안의 `data`가 가리키는 정수 객체를 참조해. **데이터 자체를 품는 게 아니라 참조를 품어.**
- ② 꼬리 노드의 `next` 는 **`None`**. "더 이상 뒤쪽 노드가 없다"는 표시야. 이게 5번의 모든 판정식의 뿌리지.
- 🔥 꼬리가 머리를 가리키면 **원형 리스트**(08-4). 그러면 `while ptr is not None` 같은 종료 조건이 영원히 안 끝나서 다른 방식이 필요해져.

**실측**: `Node` 에 리스트를 담고 원본을 수정하면 `node.data` 도 같이 바뀌어. 4일차 call by object reference 그대로야.

---

### 3. 용어 정리
- ② **머리**(head) 노드 → `A`
- ③ **꼬리**(tail) 노드 → `F`
- ④ **앞쪽**(predecessor) 노드
- ⑤ **뒤쪽**(successor) 노드
- `C`의 앞쪽은 `B`, 뒤쪽은 `D`

🔥 **결정적 약점**: **앞쪽으로 갈 수 없다.** `ptr.next` 는 있어도 `ptr.prev` 는 없거든. `D`를 보려면 **반드시 `head`부터** 걸어와야 해 → **임의 접근 불가, O(n)**

- 이게 10번(`remove_last` 의 `pre`)과 11번(`remove` 의 `ptr.next` 내다보기)이 존재하는 이유야.
- 이 약점을 없애려고 `prev` 포인터를 추가한 게 08-4의 **이중 연결 리스트**야.

---

## 🔍 PART 2 해설

### 4. 🔥 `head` 는 머리 노드가 아니다

- ① 빈 리스트일 때 `head` 는 **`None`**
- ② `head.data` = **머리 노드의 데이터**
- ③ `head.next` = **2번째 노드에 대한 참조**
- ④ **노드가 삭제되지 않아.** `head` 라는 **변수가 가리키는 곳만** 바뀔 뿐이야.

**실측**
```
head = head.next 실행 후
  원래 머리 노드 A는? old_head.data = A   ← 객체는 아직 살아 있다!
```

→ "삭제"의 정체는 **"리스트에서 그 노드를 가리키는 화살표를 끊는 것"** 이야. 아무도 안 가리키면 파이썬이 알아서 메모리를 회수(가비지 컬렉션)하고.

> 🔑 `head` 는 **리스트 밖에 있는 하나의 변수**, 머리 노드는 **Node 객체**. 이 구분이 안 되면 9~11번이 전부 흐려져.

---

### 5. 판정식은 그림에서 나온다

| | 그림 | 판정식 |
|---|---|---|
| (가) 빈 리스트 | `head → None` | `head is None` |
| (나) 1개 | `head → [A] → None` | `head.next is None` |
| (다) 2개 | `head → [A] → [B] → None` | `head.next.next is None` |
| (라) 꼬리 판별 | `p → None` | `p.next is None` |

🎯 **규칙 하나**: `next` 를 따라가다 **`None`을 몇 번째에 만나느냐**만 세면 돼. 외울 게 없어.

**`no` vs `head` 방식**
- `no` 는 **O(1)** 로 빠르고 읽기 쉬워. 하지만 `no` 가 **정확히 관리되어야만** 신뢰할 수 있지 (20번 버그가 정확히 이걸 깨뜨려!).
- `head` 따라가기는 **항상 정확**하지만 노드를 훑어야 해.
- → `no` 를 쓰되 **`no` 갱신을 절대 틀리지 않게** 하는 게 정석이야.

---

### 6. `search()` 추적

| 단계 | ptr | cnt | `== 'D'`? | 다음 |
|---|---|---|---|---|
| 1 | A | 0 | ① **False** | ② cnt=1, ptr=B |
| 2 | ③ **B** | ④ **1** | ⑤ **False** | ⑥ cnt=2, ptr=C |
| 3 | C | 2 | False | cnt=3, ptr=D |
| 4 | D | 3 | **True** | current=D, **return 3** |

- ⑦ 반환값 **3**, ⑧ `current` 는 **D**
- **+1 하는 이유**: `cnt` 는 0부터 세는 **인덱스**인데, 사람에게 보여줄 때는 "3번째"처럼 **1부터** 세는 게 자연스러우니까. 26일차 `bf_match` 의 `idx + 1` 과 같은 처리야.

**종료 조건**
- ⑨ 조건 1: **꼬리 노드까지 왔는데 못 찾음** → `while ptr is not None` 이 거짓
- ⑩ 조건 2: **찾음** → `if ptr.data == data` 가 참

**검색 실패 시 `current` 는 안 바뀌어.** `self.current = ptr` 이 `if` 블록 **안**에만 있으니까. (17번에서 다시)

---

## ✏️ PART 3 해설

### 7. `add_first`

- ① `ptr` → **삽입 전의 머리 노드 A**
- ② ③ 새 노드 `G` 의 `next` → **A**
- ④ `head` → **G**

**질문 답**
- **`ptr` 없이 한 줄로도 돼**: `self.head = Node(data, self.head)`. 파이썬은 **오른쪽을 먼저 평가**하므로 `self.head` 가 아직 옛 값(A)일 때 읽혀. 실측에서 확인했지. 교재가 `ptr` 을 쓴 건 **"삽입 전 머리 노드"라는 의미를 이름으로 드러내려는** 것.
- **순서를 바꿔도 동작**해. `self.head = Node(data, None)` 후 `self.head.next = ptr`. 다만 두 줄이 되고, 중간에 잠깐 **끊긴 상태**가 생겨서 덜 안전해.
- **빈 리스트도 분기 불필요**: `ptr = None` → `Node(data, None)` → 이게 곧 꼬리 노드 조건과 일치해. **우연이 아니라 `None` 을 "끝"으로 정한 설계 덕분**이야.

---

### 8. `add_last`

- ① `while` 종료 시 `ptr` → **꼬리 노드**
  - 조건이 `ptr.next is not None` 인 건 5번 (라)의 꼬리 판정식을 **뒤집은 것**이야. "꼬리가 아닌 동안 계속 간다."
- ② 새 노드 `G` 가 `C` 뒤에 붙어

**질문 답**
- 새 노드의 `next` 를 `None` 으로: **새 꼬리가 되니까.** 안 하면 쓰레기 값을 가리켜.
- ③ `add_first` = **O(1)**, ④ `add_last` = **O(n)** (꼬리까지 훑어야 하니까)
- **빈 리스트에 `add_first` 재사용**: `head` 가 `None` 인데 `ptr.next` 를 읽으면 `AttributeError: 'NoneType' object has no attribute 'next'`. 실측에서 확인했지.

**실측**: n=4000에서 `add_first` 는 0.03ms, `add_last` 는 20ms 수준. n에 비례하는 게 눈에 보여.

---

### 9. `remove_first`

- ① `head` → **B**
- ② **"삭제"의 정체는 참조를 끊는 것.** A 객체는 그대로 살아 있지만 리스트에서 가리키는 화살표가 없어져. 참조 카운트가 0이 되면 파이썬이 회수해.
- **노드 1개일 때도 분기 불필요**: `head.next` 가 `None` 이라 `head = None` → 자연스럽게 빈 리스트.
- 🔥 **`self.no -= 1` 의 들여쓰기가 버그야.** 20번에서.

---

### 10. `remove_last` — 두 커서

**왜 커서가 둘인가**
- ① 꼬리를 가리키는 화살표는 **맨 끝에서 2번째 노드의 `next`** 야.
- 3번에서 봤듯 **앞으로 되돌아갈 수 없으니**, 한 칸 뒤처져 따라오는 `pre` 가 필요해.

| 반복 | pre | ptr | 계속? |
|---|---|---|---|
| 시작 | A | A | ② **True** |
| 1회 | ③ **A** | ④ **B** | ⑤ **True** |
| 2회 | ⑥ **B** | ⑦ **C** | ⑧ **False (종료)** |

- ⑨ `ptr` = **꼬리 C**, ⑩ `pre` = **맨 끝에서 2번째 B**
- ⑪ `pre.next = None` → **B가 꼬리가 되고 C는 아무도 참조하지 않게 돼**

**이전 커서들과의 차이** 🔥: 18/21/23일차의 커서는 전부 **정수 인덱스**였어. `pl`, `pr`, `pa` 는 `+1`, `-1` 로 자유롭게 움직였지. 하지만 `pre`/`ptr` 은 **노드 참조**라서 **`next` 방향으로만** 갈 수 있어. 그래서 "뒤처져 따라가기"라는 기법이 필요해진 거야.

---

### 11. `remove(p)`

- ① `while` 종료 시 `ptr` → **p의 앞쪽 노드 C**
- ② `ptr.next = p.next` → C가 **E**를 가리키게 돼

**질문 답**
- **`p is self.head` 를 먼저 검사하는 이유**: 머리 노드에는 **앞쪽 노드가 없어.** `while ptr.next is not p` 를 돌리면 `head.next` 부터 시작하니 머리 자신은 절대 찾지 못하고 끝까지 가서 `return` 해버려 → **삭제가 조용히 실패**해.
- **`is` vs `==`**: `==` 는 **값**을 비교해. 같은 데이터를 가진 **다른 노드**가 있으면 엉뚱한 걸 지워. `is` 는 **같은 객체**인지 보니까 정확해. (3일차 `is` vs `==`)
- **`if ptr is None: return` 이 없으면**: `p` 가 리스트에 없을 때 `ptr` 이 `None` 이 되고 다음 반복에서 `ptr.next` → `AttributeError`
- **두 방식 비교**:
  - `pre`/`ptr` (remove_last): 변수 2개, 로직이 대칭적
  - `ptr.next` 내다보기 (remove): 변수 1개, 대신 `ptr.next` 가 `None` 일 때를 조심해야 함
  - → 취향 문제지만, **한 칸 내다보기**가 변수가 적어 더 간결해.

---

## 💻 PART 4 해설

### 12~16. 빈칸 정답

**12번**
```python
self.data = data          # ①
self.next = next          # ②
self.no = 0               # ③
self.head = None          # ④
self.current = None       # ⑤
return self.no            # ⑥
```

**13번**
```python
ptr = self.head                   # ①
while ptr is not None:            # ②
    if ptr.data == data:          # ③
        self.current = ptr        # ④
        return cnt                # ⑤
    ptr = ptr.next                # ⑥
return -1                         # ⑦
return self.search(data) >= 0     # ⑧
```

**14번**
```python
ptr = self.head                              # ①
self.head = self.current = Node(data, ptr)   # ②
if self.head is None:                        # ③
    self.add_first(data)                     # ④
while ptr.next is not None:                  # ⑤
ptr.next = self.current = Node(data, None)   # ⑥
```

**15번**
```python
if self.head is not None:                    # ①
    self.head = self.current = self.head.next    # ②
if self.head.next is None:                   # ③
while ptr.next is not None:                  # ④
    pre = ptr                                # ⑤
    ptr = ptr.next                           # ⑥
pre.next = None                              # ⑦
if p is self.head:                           # ⑧  ← is! (== 아님)
while ptr.next is not p:                     # ⑨
ptr.next = p.next                            # ⑩
```

**16번**
```python
while self.head is not None:                 # ①
    self.remove_first()                      # ②
self.current = None                          # ③
self.no = 0                                  # ④
if self.current is None or self.current.next is None:   # ⑤⑥
self.current = self.current.next             # ⑦
```

**⚠️ 15번의 ①②를 주목** — 여기서는 `self.no -= 1` 을 `if` **안**에 넣었어. 교재 코드와 다르지? 20번의 버그를 미리 고친 버전이야.

**💡 `clear()` 가 `remove_first` 를 반복하는 이유**: 노드를 하나씩 떼어내면 참조가 끊겨 파이썬이 회수해. `self.head = None` 한 줄로도 되긴 하지만(파이썬은 연쇄적으로 회수), 교재는 **의도를 명시적으로** 드러낸 거야.

---

## 🎯 PART 5 해설

### 17. 🔥🔥 표 8-1 유도

> **원리: "그 함수가 마지막으로 손댄, 리스트에 아직 남아 있는 노드"**

| 함수 | 마지막으로 손댄 노드 | 남아 있나 | → current |
|---|---|---|---|
| `add_first()` | ① **삽입한 노드** | ② **✅** | 삽입한 머리 노드 |
| `add_last()` | ③ **삽입한 노드** | ④ **✅** | ⑤ **삽입한 꼬리 노드** |
| `remove_first()` | 삭제한 머리 | ❌ | ⑥ **삭제한 뒤의 머리 노드** |
| `remove_last()` | 삭제한 꼬리 | ❌ | ⑦ **`pre` = 새 꼬리 노드** |
| `remove(p)` | 삭제한 p | ❌ | ⑧ **`ptr` = p의 앞쪽 노드** |
| `clear()` | 전부 삭제 | ❌ | ⑨ **None** |
| `next()` | ⑩ **이동한 노드** | ✅ | ⑪ **이동한 뒤 주목 노드** |
| `print()` | ⑫ **리스트를 안 바꿈** | - | ⑬ **그대로 (업데이트 안 함)** |

**실측** — 전부 원리대로 나와:
```
search('C')      → C     찾은 노드를 손댔고 남아 있음
search('Z') 실패  → A     아무 노드도 안 손댐 → 그대로
add_first('X')   → X     삽입한 노드
remove_first()   → B     머리를 지웠으니 남은 건 그 다음
remove_last()    → C     꼬리를 지웠으니 남은 건 그 앞(pre)
remove(B노드)     → A     B를 지웠으니 남은 건 그 앞(ptr)
clear()          → None  남은 노드가 없음
print()          → A     리스트를 안 바꿈
```

**추가 질문**
- **`search` 실패 시 `current` 는 안 바뀌어.** `self.current = ptr` 이 `if ptr.data == data:` **블록 안**에만 있거든. 원리로 보면 "아무 노드도 손대지 않았으니 그대로".
- ⑭ **`print` 계열은 리스트를 변경하지 않는 읽기 전용 함수**라서. 읽기가 상태를 바꾸면 부작용(side effect)이 돼 (19번).

> 🎯 **11줄 암기 → 한 문장.** 이게 오늘 노트북에서 제일 남기고 싶은 거야.

---

### 18. 이터레이터

| | 역할 |
|---|---|
| `LinkedList.__iter__` | ① **이터레이터 객체를 새로 만들어 반환** |
| `LinkedListIterator` | ② **실제 순회 상태(어디까지 갔나)를 들고 하나씩 꺼냄** |

**빈칸 정답**
```python
self.current = head          # ①
return self                  # ②
if self.current is None:     # ③
    raise StopIteration      # ④
data = self.current.data     # ⑤
self.current = self.current.next   # ⑥
return LinkedListIterator(self.head)   # ⑦
```

**🔥 왜 별도 클래스로 분리하나**
`LinkedList` 자신이 `__next__` 와 순회 위치를 가지면, **같은 리스트를 두 곳에서 동시에 순회할 수 없어.** 중첩 for문(`for a in lst: for b in lst:`)이 서로의 위치를 덮어써서 망가지지. 이터레이터를 매번 새로 만들면 **각자 독립된 커서**를 갖게 돼.

- 실측에서 중첩 for가 **16개 쌍**(4×4)을 정상적으로 만들어냈지.
- ③ `LinkedListIterator.current` 와 `LinkedList.current` 는 **완전히 다른 것**이야. 이름만 같아. 19번에서 확인.

---

### 19. 이터레이터는 `current` 를 안 건드린다

**실측**
```
search('C') 후 current = C
for 순회 후    current = C   ← 그대로! ✅

나쁜 이터레이터 순회 후 current = D  ← 꼬리로 끌려갔다 ❌
```

- **순회는 "읽기" 작업**이야. 읽기가 객체의 상태를 바꾸면 **부작용**이 되고, 사용자가 애써 설정한 주목 노드가 날아가.
- 25일차 `sorted()` vs `.sort()`, 4일차 in-place vs 새 객체 논의와 같은 계열이야. **"이 연산이 원본을 바꾸는가"** 는 항상 따져야 할 질문.

---

## 🐛 PART 6 해설

### 20. 🔥🔥 교재 코드의 진짜 버그

- ① **빈 리스트인데도 `no` 가 1 감소해서 음수가 돼.**

**실측**
```
초기: no = 0
1회 후: no = -1   ← 음수! 🔥
3회 후: no = -3
len(e) → ValueError: __len__() should return >= 0
```

🔥 **에러가 즉시 나지 않고, 나중에 `len()` 을 부를 때 엉뚱한 곳에서 터져.** 이게 이 버그가 악질인 이유야 — 원인과 증상이 멀리 떨어져 있거든.

**비대칭 확인**
```python
# remove_last
if self.head is not None:
    ...
        self.no -= 1        # if 안쪽 ✅

# remove_first
if self.head is not None:
    ...
self.no -= 1                # if 바깥 ❌
```

**고치는 법**: `self.no -= 1` 을 **한 단계 들여쓰기** 하면 끝. 딱 4칸.

**이번 달 교재 버그 모음**
| 일차 | 버그 |
|---|---|
| 22일차 14번 | `insertion_sort` 의 `while j > 0` (호출 맥락 덕에 우연히 안전) |
| 28일차 12번 | `bm_match` 의 `pt` 초기화 누락 (for 변수 누출에 의존) |
| **29일차 (오늘)** | `remove_first` 의 `no -= 1` 들여쓰기 |

> 🔑 **교재 코드도 의심하며 읽어라.** 이게 이번 달 내내 반복된 교훈이야.

---

### 21. `add_last` 가 느린 이유

- ① **0칸** (`add_first` 로 위임) ② **0칸** ③ **n-2칸**
- ④ **(n-1)(n-2)/2** ⑤ **O(n²)**

💡 1번째는 `add_first` 로 넘어가고 2번째는 노드가 1개뿐이라 while이 안 돌아. 그래서 `n(n-1)/2` 가 아니라 **`(n-1)(n-2)/2`** 야. 어느 쪽이든 **n²에 비례**하는 건 같지.

**실측** — 정확히 `(n-1)(n-2)/2` 와 일치:
```
     n | 총 스캔    | (n-1)(n-2)/2
   100 |    4,851 |        4,851
   200 |   19,701 |       19,701
   400 |   79,401 |       79,401
   800 |  318,801 |      318,801
```

**고치는 법**: 🔥 **꼬리 노드를 `tail` 필드에 기억해두면** `add_last` 가 **O(1)** 이 돼. 실측에서 n=8000일 때 **184배** 차이가 나.

- 이게 08-4 **원형 이중 연결 리스트**로 가는 동기 중 하나야. 원형이면 `head.prev` 가 곧 꼬리라서 별도 `tail` 도 필요 없어지지.
- 💡 **"자주 쓰는 값을 미리 계산해 저장한다"** — 25일차 도수 분포표, 27일차 skip 표, 28일차 이동량 표와 **완전히 같은 발상**이야.

---

### 22. 배열 vs 연결 리스트

| 연산 | 배열 | 연결 리스트 |
|---|---|---|
| i번째 접근 | O(1) | ① **O(n)** |
| 맨 앞 삽입 | ② **O(n)** | ③ **O(1)** |
| 맨 끝 삽입 | O(1) 평균 | ④ **O(n)** / ⑤ **O(1)** (tail 유지) |
| 맨 앞 삭제 | ⑥ **O(n)** | ⑦ **O(1)** |
| 검색 | O(n) | ⑧ **O(n)** |
| 메모리 | 포인터 1개 | ⑨ **데이터 + next + 객체 오버헤드** |

**실측 (n=1000)**
```
Node 1개 ≈ 136 바이트
노드 1000개 ≈ 132.8 KB
파이썬 list 1000개 = 7.9 KB
→ 약 17배 차이 🔥
```

**최종 질문 답**

1. **연결 리스트가 확실히 유리한 상황**: **맨 앞(또는 이미 참조를 갖고 있는 위치)에서 잦은 삽입·삭제.** 배열은 O(n)이지만 연결 리스트는 O(1)이야. 실측에서도 맨 앞 삽입은 연결 리스트가 이겼지.

2. **`list.append()` 의 최악**: 미리 확보해둔 여유 공간이 다 차면 **더 큰 배열을 새로 만들어 전부 복사**해야 해 → 그 한 번은 **O(n)**. 다만 용량을 배로 늘리므로 **평균(amortized)으로는 O(1)** 이야.

3. **08-3이 겨냥하는 약점**: 노드마다 **객체를 생성·소멸하는 비용**이야. 22번 실측의 메모리 17배 차이가 그 증거고, 파이썬에서 객체 생성은 특히 비싸. 배열 안에 노드를 미리 만들어두고 **인덱스(커서)로 잇는** 게 08-3의 해법이지.

---

## 📌 핵심 3줄 요약

1. **연결 리스트는 "데이터를 옮기지 않고 화살표만 바꾼다".** 그래서 맨 앞 삽입·삭제가 O(1)이야. 대신 앞쪽으로 되돌아갈 수 없어서 임의 접근이 O(n)이고, `remove_last` 는 `pre` 라는 뒤처진 커서가 필요해져.
2. **표 8-1은 외우는 게 아니라 유도하는 것.** "그 함수가 마지막으로 손댄, 리스트에 남아 있는 노드" 한 문장이면 11줄이 전부 나와. 판정식(`head is None`, `head.next is None`...)도 그림에서 `None` 을 몇 번째에 만나는지만 세면 돼.
3. **`head` 는 머리 노드가 아니라 머리 노드에 대한 참조다.** "삭제"란 노드를 지우는 게 아니라 **가리키는 화살표를 끊는 것**이고, 아무도 안 가리키면 파이썬이 회수해. 이 구분이 되면 삽입·삭제 코드가 전부 자연스럽게 읽혀.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 3, 5, 7, 9, 12, 22번)**: 전원 필수
  - **5번 판정식**과 **7번 add_first** 는 종이에 직접 그릴 것. 이거 없이 코드로 가면 다 외우게 돼
  - **3번 "앞으로 못 간다"** 를 잡아야 10~11번이 이해돼
- 🟡 **(R-2, 6, 8, 13, 14, 16, 18, 19, 21번)**: 팀 목표선
  - **8번 add_last** 에서 O(n)임을 실측으로 볼 것 → 21번의 O(n²)로 이어짐
  - **18번 이터레이터**는 중첩 for가 왜 되는지까지 설명할 수 있어야 함
- 🔴 **(4, 10, 11, 15, 17, 20번)**: 도전
  - **17번이 오늘 최고 난도이자 최대 수확** 🔥🔥 — 표 8-1을 한 문장으로 무너뜨리는 문제
  - **4번 `head` 정체**와 **20번 들여쓰기 버그**는 짝으로 볼 것
  - **15번**은 오늘 코드 중 가장 길어. 10~11번 그림 없이는 못 채워
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 22번)

## 🔗 오늘 회수된 개념들

- **4일차 call by object reference** → `next` 는 참조, `head = head.next` 는 화살표 이동 (R-2, 4번)
- **3일차 `is` vs `==`** → `remove` 에서 `p is self.head` (11번)
- **12일차 선형 검색** → `search()` 가 그대로 선형 검색 (6번)
- **18일차 들여쓰기 버그** → `no -= 1` 위치 (20번)
- **21·23일차 투 포인터** → `pre`/`ptr` 은 인덱스가 아니라 **노드 참조** (10번)
- **22일차 14번, 28일차 12번 교재 버그** → 오늘도 하나 더 (20번)
- **25·27·28일차 "표 미리 만들기"** → `tail` 포인터를 유지하는 발상 (21번)
- **4일차 in-place vs 새 객체** → 이터레이터가 `current` 를 안 건드리는 이유 (19번)

---

> **다음 진도 (30일차)**: 08-3 **커서를 이용한 연결 리스트**
>
> 교재 342p: **"노드를 삽입·삭제할 때마다 내부에서 노드용 인스턴스를 생성하고 소멸합니다. 이때 메모리를 확보하고 해제하는 데 쓰는 비용을 결코 무시할 수 없습니다."**
>
> 오늘은 노드를 **객체**로 만들었지만, 08-3은 **배열 안의 원소**로 만들어. 그럼 `next` 가 참조가 아니라 **인덱스(커서)** 가 되지.
> 배열의 장점(메모리 효율)과 연결 리스트의 장점(삽입·삭제 O(1))을 **둘 다** 가지려는 시도야.
> 그리고 08-4에서 **원형 이중 연결 리스트**로 오늘의 두 약점(앞으로 못 감, `add_last` 가 O(n))을 한 번에 해결해.